# Kepler exoplanet classification - Notebook 01

**Data exploration, leakage analysis, and cleaning**

Author: Atilla Ahmed

---

## Abstract 
This notebook prepares the NASA Kepler cumulative Object of Interest (KOI) table for downstream deep learning experiments. We characterize the dataset through exploratory analysis, identify and quantify three sources of potential label leakage that inflate accuracy in many published studies on this data, and produce clean stratified train/validation/test splits saved as parquet files. The output is three feature sets leaky, semi-leaky, and leak-free that allow us to measure how much reported model performance depends on label-adjacent features rather than genuine astrophysical signal.

## Table of contents

1. [Introduction](#1-introduction)
2. [Data loading and first look](#2-data-loading-and-first-look)
3. [Exploratory data analysis](#3-exploratory-data-analysis)
4. [Data leakage analysis](#4-data-leakage-analysis)
5. [Data cleaning](#5-data-cleaning)
6. [Train / validation / test splits](#6-train--validation--test-splits)
7. [Summary and next steps](#7-summary-and-next-steps)

## 1. Introduction
The NASA Kepler mission and its dataset of Kepler Objects of Interest (KOIs) form the empirical foundation for modern exoplanet science. Before we can meaningfully apply deep learning to this data, we must understand what the dataset represents, how it was generated, and what real-world problem our classifier is being asked to solve. This section provides that context in four parts: a brief history of the mission, a description of the transit method used to detect the signals, a formal problem statement, and the specific objectives of this notebook.

### 1.1. The Kepler mission
Тhe Kepler Space Telescope, launched by NASA in March 2009, was designed with a single scientific objective: to determine what fraction of stars in the Milky Way host Earth-sized planets in the habitable zone. Between 2009 and 2013, the telescope continuously monitored the brightness of approximately 150,000 main-sequence stars in a fixed field of view in the constellations Cygnus and Lyra.

### 1.2. Transit method for exoplanet detection
Kepler detects exoplanets through the *transit method*: when a planet passes in front of its host star from our line of sight, the star's brightness dips by a small fraction typically 0.01% to 1% for the duration of the transit. The depth of the dip is proportional to the square of the ratio of planet radius to stellar radius, and the interval between successive dips gives the orbital period.

### 1.3. Problem statement

Given the physical and observational parameters of a Kepler Object of Interest orbital period, transit depth, planet radius, stellar temperature, transit duration, and roughly 20 other features derived from photometric measurements we want to predict its true nature: a *confirmed* exoplanet, an unconfirmed *candidate* still under investigation, or a *false positive* caused by non-planetary phenomena. This is a three-class classification problem on the Kepler cumulative KOI table, with 9,564 rows and 153 candidate features.

### 1.4. Notebook objectives

By the end of this notebook we will have:

1. A characterization of the dataset through targeted exploratory analysis focused on features with clear astrophysical meaning.
2. An explicit identification of three types of label leakage present in the KOI table and a quantification of their impact.
3. Three curated feature sets leaky, semi-leaky, and leak-free that will allow subsequent modelling notebooks to distinguish genuine predictive skill from artefacts of label construction.
4. Cleaned stratified train, validation, and test splits saved as parquet files for use in downstream training.

## 2. Data loading and first look

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path("../data/raw/kepler_data.csv")
PROCESSED_PATH = Path("../data/processed")
FIGURES_PATH = Path("../reports/figures")

print(f"Data file exists: {DATA_PATH.exists()}")

Data file exists: True


### 2.1. Loading the dataset

In [4]:
planets = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {planets.shape[0]:,} rows x {planets.shape[1]} columns")
print(f"Memory usage: {planets.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Shape: 9,564 rows x 153 columns
Memory usage: 24.5 MB


In [5]:
planets.head()

,kepid,kepoi_name,kepler_name,ra,ra_err,ra_str,dec,dec_err,dec_str,koi_gmag,koi_gmag_err,koi_rmag,koi_rmag_err,koi_imag,koi_imag_err,koi_zmag,koi_zmag_err,koi_jmag,koi_jmag_err,koi_hmag,koi_hmag_err,koi_kmag,koi_kmag_err,koi_kepmag,koi_kepmag_err,...,koi_dikco_msky,koi_dikco_msky_err,koi_comment,koi_vet_date,koi_tce_plnt_num,koi_tce_delivname,koi_datalink_dvs,koi_disp_prov,koi_parm_prov,koi_time0,koi_time0_err1,koi_time0_err2,koi_datalink_dvr,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,koi_fpflag_ec,koi_insol,koi_insol_err1,koi_insol_err2,koi_srho,koi_srho_err1,koi_srho_err2,koi_fittype,koi_score
0,10797460,K00752.01,Kepler-227 b,291.9342,0.0,19h27m44.22s,48.1417,0.0,+48d08m29.9s,15.890,NaN,15.270,NaN,15.114,NaN,15.006,NaN,14.082,0.025,13.751,0.030,13.648,0.054,15.347,NaN,...,0.320,0.160,NO_COMMENT,2018-08-16,1.0,q1_q17_dr25_tce,010/010797/010797460/dv/kplr010797460-001-2016...,q1_q17_dr25_sup_koi,q1_q17_dr25_koi,2.4550e+06,0.0022,-0.0022,010/010797/010797460/dv/kplr010797460-20160209...,0,0,0,0,93.59,29.45,-16.65,3.2080,0.3317,-1.0999,LS+MCMC,1.000
1,10797460,K00752.02,Kepler-227 c,291.9342,0.0,19h27m44.22s,48.1417,0.0,+48d08m29.9s,15.890,NaN,15.270,NaN,15.114,NaN,15.006,NaN,14.082,0.025,13.751,0.030,13.648,0.054,15.347,NaN,...,0.500,0.450,NO_COMMENT,2018-08-16,2.0,q1_q17_dr25_tce,010/010797/010797460/dv/kplr010797460-002-2016...,q1_q17_dr25_sup_koi,q1_q17_dr25_koi,2.4550e+06,0.0035,-0.0035,010/010797/010797460/dv/kplr010797460-20160209...,0,0,0,0,9.11,2.87,-1.62,3.0237,2.2049,-2.4964,LS+MCMC,0.969
2,10811496,K00753.01,NaN,297.0048,0.0,19h48m01.16s,48.1341,0.0,+48d08m02.9s,15.943,NaN,15.390,NaN,15.220,NaN,15.166,NaN,14.254,0.028,13.900,0.033,13.826,0.058,15.436,NaN,...,0.027,0.074,DEEP_V_SHAPED,2018-08-16,1.0,q1_q17_dr25_tce,010/010811/010811496/dv/kplr010811496-001-2016...,q1_q17_dr25_sup_koi,q1_q17_dr25_koi,2.4550e+06,0.0006,-0.0006,010/010811/010811496/dv/kplr010811496-20160209...,0,0,0,0,39.30,31.04,-10.49,7.2956,35.0329,-2.7545,LS+MCMC,0.000
3,10848459,K00754.01,NaN,285.5346,0.0,19h02m08.31s,48.2852,0.0,+48d17m06.8s,16.100,NaN,15.554,NaN,15.382,NaN,15.266,NaN,14.326,0.035,13.911,0.042,13.809,0.048,15.597,NaN,...,0.276,0.076,MOD_ODDEVEN_DV---MOD_ODDEVEN_ALT---DEEP_V_SHAPED,2018-08-16,1.0,q1_q17_dr25_tce,010/010848/010848459/dv/kplr010848459-001-2016...,q1_q17_dr25_sup_koi,q1_q17_dr25_koi,2.4550e+06,0.0001,-0.0001,010/010848/010848459/dv/kplr010848459-20160209...,0,1,0,0,891.96,668.95,-230.35,0.2208,0.0092,-0.0184,LS+MCMC,0.000
4,10854555,K00755.01,Kepler-664 b,288.7549,0.0,19h15m01.17s,48.2262,0.0,+48d13m34.3s,16.015,NaN,15.468,NaN,15.292,NaN,15.241,NaN,14.366,0.033,14.064,0.047,13.952,0.047,15.509,NaN,...,0.070,0.200,NO_COMMENT,2018-08-16,1.0,q1_q17_dr25_tce,010/010854/010854555/dv/kplr010854555-001-2016...,q1_q17_dr25_sup_koi,q1_q17_dr25_koi,2.4550e+06,0.0011,-0.0011,010/010854/010854555/dv/kplr010854555-20160209...,0,0,0,0,926.16,874.33,-314.24,1.9864,2.7114,-1.7454,LS+MCMC,1.000


### 2.3. Column types and non-null counts

We inspect the dtypes and non-null counts to understand which columns are numeric, categorical, and how sparse each column is.

In [6]:
planets.info(verbose=False, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9564 entries, 0 to 9563
Columns: 153 entries, kepid to koi_score
dtypes: float64(127), int64(6), object(20)
memory usage: 11.2+ MB


### 2.4. Functional column groups

Not all 153 columns serve the same purpose. Before deciding what to use as model input, we categorize them into functional groups. This categorization drives the leakage analysis in Section 4 and the final feature selection in Section 5.

In [11]:
target_col = "koi_disposition"

identifier_cols = ["kepid", "kepoi_name", "kepler_name"]

coord_cols = ["ra", "ra_str", "dec", "dec_str"]

target_related_cols = [
    "koi_pdisposition",
    "koi_score",
    "koi_disp_prov",
    "koi_vet_stat",
    "koi_vet_date",
    "koi_comment",
]

fp_flag_cols = [
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
]

metadata_cols = [
    "koi_delivname",
    "koi_quarters",
    "koi_tce_plnt_num",
    "koi_tce_delivname",
    "koi_datalink_dvs",
    "koi_datalink_dvr",
    "koi_parm_prov",
    "koi_sparprov",
    "koi_limbdark_mod",
    "koi_trans_mod",
    "koi_fittype",
]
error_cols = [c for c in planets.columns if c.endswith("_err") or c.endswith("_err1") or c.endswith("_err2")]
categorized = set(
    [target_col]
    + identifier_cols
    + coord_cols
    + target_related_cols
    + fp_flag_cols
    + metadata_cols
    + error_cols
)

physical_cols = [c for c in planets.columns if c not in categorized]

print(f"Target column:        1  ({target_col})")
print(f"Identifier columns:   {len(identifier_cols)}")
print(f"Coordinate columns:   {len(coord_cols)}")
print(f"Target-related:       {len(target_related_cols)}")
print(f"False positive flags: {len(fp_flag_cols)}")
print(f"Metadata columns:     {len(metadata_cols)}")
print(f"Error columns:        {len(error_cols)}")
print(f"Physical measurements: {len(physical_cols)}")
print(f"Total:                {1 + len(identifier_cols) + len(coord_cols) + len(target_related_cols) + len(fp_flag_cols) + len(metadata_cols) + len(error_cols) + len(physical_cols)}")


Target column:        1  (koi_disposition)
Identifier columns:   3
Coordinate columns:   4
Target-related:       6
False positive flags: 4
Metadata columns:     11
Error columns:        68
Physical measurements: 56
Total:                153


### 2.5. Target variable overview

The target `koi_disposition` takes three values: `CONFIRMED` for verified exoplanets, `CANDIDATE` for KOIs still under investigation, and `FALSE POSITIVE` for signals attributed to non-planetary phenomena. We examine the class distribution to assess balance.

In [13]:
class_counts = planets[target_col].value_counts()
class_pct = planets[target_col].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_pct.round(1),
})

print(target_summary)

                 count  percentage
koi_disposition                   
FALSE POSITIVE    4839        50.6
CONFIRMED         2747        28.7
CANDIDATE         1978        20.7


The class distribution shows moderate imbalance. `FALSE POSITIVE` dominates at 50.6%, while `CANDIDATE` is the smallest class at 20.7%. A trivial classifier that always predicts `FALSE POSITIVE` would achieve 50.6% accuracy, which sets a hard lower bound for meaningful models.

### 2.6. Key findings from initial exploration

- The KOI table contains **9,564 rows and 153 columns**, of which 56 are physical measurements potentially usable as features, 68 are error terms for those measurements, and the remainder are identifiers, coordinates, and provenance metadata.
- The target `koi_disposition` has **three classes with moderate imbalance** (50.6% / 28.7% / 20.7%).